# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suha-2004/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
### Unit of analysis

One row represents one content item/page for a client.

The dataset contains 30,000 rows and 44 columns. Each row contains content characteristics and search-performance signals.

### Time window

The dataset contains 90-day aggregate performance measures and previous/current 30-day windows. The available time-window fields include `impressions_90d`, `clicks_90d`, `sessions_90d`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, and `sessions_prev_30d`.

The analysis therefore uses the time windows represented by these dataset fields rather than assuming exact calendar dates that are not explicitly provided in the dataset.

In [1]:
!git clone https://github.com/suha-2004/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 249, done.
remote: Counting objects: 100% (249/249), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 249 (delta 128), reused 102 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (249/249), 3.07 MiB | 9.48 MiB/s, done.
Resolving deltas: 100% (128/128), done.


In [2]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [3]:
import os

print(os.getcwd())
print(os.listdir("./data/raw"))

/content/flyrank-ml-internship
['content_refresh_anonymized.csv']


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

Shape: (30000, 44)
Number of rows: 30000
Number of columns: 44


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
### Feature fields

The final feature set used for the modeling lane is:

- `impressions_prev_30d` — previous 30-day impressions
- `clicks_prev_30d` — previous 30-day clicks
- `sessions_prev_30d` — previous 30-day sessions
- `content_age_days` — age of the content
- `days_since_last_update` — time since the content was last updated
- `ctr` — click-through rate
- `engagement_rate` — engagement rate

These features are intended to represent information available before the prediction point.

### Label

The modeling target/proxy is `avg_position`, representing average search ranking position. Lower values represent better search position.

### Context fields

Context fields include `content_type`, `main_intent`, `competition_level`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`, and `trend_direction`.

These fields help describe the content and its search context.

### Excluded fields

`impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d` are excluded from the final feature set because they represent the outcome/current-period window rather than the historical information used for prediction.

`content_id` and `client_id` are identifiers and are not used as predictive features.

Other outcome-derived or target-like fields are excluded when they would introduce leakage or would not be available at the prediction point.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify dataset shape and columns

print("Dataset shape:", df.shape)

print("\nFirst 10 columns:")
print(df.columns[:10].tolist())

print("\nNumber of unique content IDs:")
print(df["content_id"].nunique())

print("\nNumber of unique clients:")
print(df["client_id"].nunique())

Dataset shape: (30000, 44)

First 10 columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']

Number of unique content IDs:
30000

Number of unique clients:
32


In [7]:
window_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("Available time-window columns:")
print([col for col in window_cols if col in df.columns])

Available time-window columns:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']


In [8]:
check_cols = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "engagement_rate",
    "avg_position"
]

missing_check = df[check_cols].isna().sum()

print("Missing values:")
print(missing_check)

Missing values:
impressions_prev_30d      0
clicks_prev_30d           0
sessions_prev_30d         0
content_age_days          0
days_since_last_update    0
ctr                       0
engagement_rate           0
avg_position              0
dtype: int64


In [9]:
outcome_period_cols = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

print("Outcome-period columns found:")
print([c for c in outcome_period_cols if c in df.columns])

Outcome-period columns found:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d']


In [10]:
feature_cols = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "engagement_rate"
]

print("Final feature columns:")
print(feature_cols)

print("\nNumber of features:", len(feature_cols))

print("\nFeature data shape:")
print(df[feature_cols].shape)

Final feature columns:
['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'engagement_rate']

Number of features: 7

Feature data shape:
(30000, 7)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
### Data limitations

The dataset does not provide a complete balanced history for every content item and client, so comparisons across rows should be interpreted carefully.

The available fields contain aggregated 90-day and 30-day windows rather than a complete daily history. Therefore, the data cannot fully describe every change that happened during the observation period.

The previous 30-day and current/last 30-day windows are related time periods. They should not be treated as independent observations without considering their temporal relationship.

The dataset also cannot establish causation. Observed relationships between search-performance signals and ranking outcomes should be treated as directional evidence and decision-support rather than proof that changing one feature will cause a ranking improvement.

Early history may have different data availability from later history, so missingness and coverage should be considered when interpreting results.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("ML-04 DATA CONTRACT SELF-CHECK")
print("==============================")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("Feature columns:", len(feature_cols))
print("Target/proxy:", "avg_position")

print("Outcome-period columns excluded:",
      set(outcome_period_cols).isdisjoint(set(feature_cols)))

print("Data contract checks completed.")

ML-04 DATA CONTRACT SELF-CHECK
Rows: 30000
Columns: 44
Feature columns: 7
Target/proxy: avg_position
Outcome-period columns excluded: True
Data contract checks completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.